# IRT Exploration
Visualise the 3PL Item Response Theory curves and theta estimation.

In [ ]:
import sys
sys.path.insert(0, '../../')
import numpy as np
import matplotlib.pyplot as plt
from backend.irt.model import IRTModel

In [ ]:
model = IRTModel()
theta_range = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ICC curves
ax = axes[0]
for b in [-2, -1, 0, 1, 2]:
    p = [model.probability(t, b) for t in theta_range]
    ax.plot(theta_range, p, label=f'b={b}')
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0.25, color='gray', linestyle=':', alpha=0.5, label='Guessing (c=0.25)')
ax.set_title('Item Characteristic Curves (3PL)')
ax.set_xlabel('Theta (ability)')
ax.set_ylabel('P(correct)')
ax.legend()
ax.grid(alpha=0.3)

# Fisher Information
ax = axes[1]
for b in [-2, 0, 2]:
    fi = [model.fisher_information(t, b) for t in theta_range]
    ax.plot(theta_range, fi, label=f'b={b}')
ax.set_title('Fisher Information')
ax.set_xlabel('Theta (ability)')
ax.set_ylabel('Information')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Simulate theta estimation after N responses
np.random.seed(42)
TRUE_THETA = 1.0

responses = []
thetas = []
questions = [{'b': b, 'a': 1.0, 'c': 0.25} for b in np.linspace(-2, 2, 20)]

for i, q in enumerate(questions):
    p = model.probability(TRUE_THETA, q['b'])
    correct = np.random.rand() < p
    responses.append({**q, 'correct': bool(correct)})
    est_theta = IRTModel().update_theta(responses)
    thetas.append(est_theta)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(thetas)+1), thetas, 'b-o', markersize=4, label='Estimated theta')
plt.axhline(TRUE_THETA, color='r', linestyle='--', label=f'True theta = {TRUE_THETA}')
plt.xlabel('Number of questions answered')
plt.ylabel('Theta estimate')
plt.title('Theta Convergence with MLE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final estimate: {thetas[-1]:.3f} (true: {TRUE_THETA})')

In [ ]:
# CAT simulation: compare random vs adaptive question selection
from backend.irt.cat import select_cat_question

question_bank = [{'id': str(i), 'b': b, 'a': 1.0, 'c': 0.25}
                 for i, b in enumerate(np.linspace(-3, 3, 50))]

def simulate(true_theta, adaptive=True, n_questions=15):
    responses = []
    thetas = []
    model = IRTModel()
    seen = set()

    for _ in range(n_questions):
        if adaptive:
            q = select_cat_question(model.theta, question_bank, seen)
        else:
            available = [q for q in question_bank if q['id'] not in seen]
            q = available[np.random.randint(len(available))] if available else None
        if not q:
            break
        seen.add(q['id'])
        p = model.probability(true_theta, q['b'])
        correct = np.random.rand() < p
        responses.append({**q, 'correct': bool(correct)})
        model.update_theta(responses)
        thetas.append(model.theta)
    return thetas

np.random.seed(0)
TRUE_THETA = 0.8
adaptive_thetas = simulate(TRUE_THETA, adaptive=True)
random_thetas = simulate(TRUE_THETA, adaptive=False)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(adaptive_thetas)+1), adaptive_thetas, 'g-o', markersize=5, label='CAT (adaptive)')
plt.plot(range(1, len(random_thetas)+1), random_thetas, 'b--s', markersize=5, label='Random selection')
plt.axhline(TRUE_THETA, color='r', linestyle='--', label=f'True theta = {TRUE_THETA}')
plt.xlabel('Questions answered')
plt.ylabel('Theta estimate')
plt.title('CAT vs Random: Convergence Speed')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()